# Building a Local Data Warehouse with DuckDB

**BINFX410 — Chapter 10, Notebook 3 of 5**

---

In this notebook we build a fully functional **data warehouse** using **DuckDB** — an in-process, OLAP-optimized SQL database that runs entirely on your laptop with no server required. We implement a classic **star schema** for genomics data, load it via an ETL pipeline, and run real analytical queries.

**Prerequisites:** Run `01_introduction_and_setup.ipynb` first to generate `./raw_data/`.

## Learning Objectives

1. Explain star schema design: fact tables, dimension tables, surrogate keys.
2. Understand the difference between OLTP and OLAP workloads.
3. Build a DuckDB data warehouse from a CSV source.
4. Write multi-table analytical SQL (window functions, CTEs, aggregations).
5. Use `EXPLAIN` to understand query execution plans.

## 1. Data Warehouse Concepts

### 1.1 OLTP vs OLAP

```
OLTP (Online Transaction Processing)    OLAP (Online Analytical Processing)
─────────────────────────────────────   ────────────────────────────────────
Purpose: Record individual events        Purpose: Analyze historical patterns
Queries: Many small reads/writes         Queries: Few large scans
Tables:  3NF normalized, many joins      Tables:  Denormalized, few joins
Example: INSERT INTO variant_calls ...   Example: SELECT SUM(depth) ...
Latency: Milliseconds                    Latency: Seconds to minutes
Rows/q:  1–100                           Rows/q:  Millions
Tool:    PostgreSQL, MySQL               Tool:    DuckDB, Snowflake, BigQuery
```

### 1.2 Star Schema

```
                        ┌─────────────────┐
                        │   dim_date      │
                        │─────────────────│
                        │ date_key (PK)   │
                        │ full_date       │
                        │ year, month     │
                        │ quarter, week   │
                        └────────┬────────┘
                                 │
┌──────────────┐        ┌────────▼────────┐        ┌──────────────────┐
│ dim_samples  │        │  fact_variants  │        │    dim_genes     │
│──────────────│        │─────────────────│        │──────────────────│
│ sample_key   ├───────►│ sample_key      │◄───────┤ gene_key         │
│ sample_id    │        │ gene_key        │        │ gene_id          │
│ patient_id   │        │ call_key        │        │ gene_name        │
│ tissue_type  │        │ date_key        │        │ chromosome       │
│ project_id   │        │─────────────────│        │ biotype          │
│ mean_coverage│        │ af_tumor    (M) │        │ is_cancer_gene   │
└──────────────┘        │ depth       (M) │        └──────────────────┘
                        │ quality_score(M)│
                        │ signal_strength(M)│
                        └────────┬────────┘
                                 │
                        ┌────────▼────────┐
                        │ fact_variant_   │
                        │    calls        │
                        │─────────────────│
                        │ call_key (PK)   │
                        │ sample_key      │
                        │ date_key        │
                        │ caller          │
                        │ filter_status   │
                        │ variant_count(M)│
                        │ mean_quality (M)│
                        └─────────────────┘
```

**Key concepts:**
- **(M)** = measure (numeric, aggregatable)
- **dim_** tables describe *who/what/when/where*
- **fact_** tables describe *events* and hold measures
- Surrogate keys (`_key`) decouple warehouse from source system IDs

### 1.3 Why DuckDB?

DuckDB is a modern, embedded OLAP database:

| Feature | Benefit |
|---------|--------|
| **In-process** | No server, no configuration, single file |
| **Columnar storage** | Only reads columns needed for a query |
| **Vectorized execution** | Processes data in batches for CPU cache efficiency |
| **Parallel query** | Automatically uses all CPU cores |
| **SQL-complete** | Full window functions, CTEs, UNNEST, LIST aggregates |
| **Parquet native** | Can query Parquet/CSV directly, no import needed |

## 2. Setup — Create the Warehouse Database

In [1]:
import os
import duckdb
import pandas as pd

os.makedirs('./warehouse', exist_ok=True)

# Create a persistent DuckDB database file
DB_PATH = './warehouse/genomics.duckdb'

# Remove old database if re-running from scratch
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f'Removed existing database: {DB_PATH}')

con = duckdb.connect(DB_PATH)
print(f'DuckDB {duckdb.__version__} database created: {DB_PATH}')

DuckDB 1.4.3 database created: ./warehouse/genomics.duckdb


## 3. Create Dimension Tables

Dimension tables are small, descriptive tables. They change infrequently (Slowly Changing Dimensions). We create them first because fact tables reference them via foreign keys.

In [2]:
# ── dim_date ──────────────────────────────────────────────────────────────────
# A date dimension is the most commonly joined dimension in any warehouse.
# Pre-computing date attributes avoids expensive EXTRACT() calls at query time.

con.execute("""
CREATE TABLE dim_date (
    date_key     INTEGER PRIMARY KEY,   -- YYYYMMDD integer key for fast joining
    full_date    DATE    NOT NULL,
    year         INTEGER NOT NULL,
    quarter      INTEGER NOT NULL,      -- 1-4
    month        INTEGER NOT NULL,      -- 1-12
    month_name   VARCHAR NOT NULL,
    week_of_year INTEGER NOT NULL,
    day_of_month INTEGER NOT NULL,
    day_of_week  INTEGER NOT NULL,      -- 0=Sunday
    day_name     VARCHAR NOT NULL,
    is_weekend   BOOLEAN NOT NULL
)
""")

# Populate dim_date for 2018-01-01 through 2025-12-31
con.execute("""
INSERT INTO dim_date
SELECT
    CAST(strftime(d, '%Y%m%d') AS INTEGER)  AS date_key,
    d                                        AS full_date,
    YEAR(d)                                  AS year,
    QUARTER(d)                               AS quarter,
    MONTH(d)                                 AS month,
    strftime(d, '%B')                        AS month_name,
    WEEKOFYEAR(d)                            AS week_of_year,
    DAY(d)                                   AS day_of_month,
    DAYOFWEEK(d)                             AS day_of_week,
    strftime(d, '%A')                        AS day_name,
    DAYOFWEEK(d) IN (0, 6)                   AS is_weekend
FROM generate_series(
    DATE '2018-01-01',
    DATE '2025-12-31',
    INTERVAL '1 day'
) t(d)
""")

count = con.execute('SELECT COUNT(*) FROM dim_date').fetchone()[0]
print(f'dim_date populated: {count:,} rows')
con.execute("SELECT * FROM dim_date WHERE full_date = '2022-06-15'").df()

dim_date populated: 2,922 rows


,date_key,full_date,year,quarter,month,month_name,week_of_year,day_of_month,day_of_week,day_name,is_weekend
0,20220615,2022-06-15,2022,2,6,June,24,15,3,Wednesday,False


In [3]:
# ── dim_samples ───────────────────────────────────────────────────────────────
con.execute("""
CREATE TABLE dim_samples (
    sample_key         INTEGER PRIMARY KEY,  -- surrogate key
    sample_id          INTEGER NOT NULL,     -- natural/business key from source
    patient_id         INTEGER,
    tissue_type        VARCHAR,
    sequencing_platform VARCHAR,
    library_prep       VARCHAR,
    project_id         VARCHAR,
    collection_date    DATE,
    mean_coverage      DECIMAL(8,1),
    pct_bases_20x      DECIMAL(6,4),
    dw_created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

con.execute("""
INSERT INTO dim_samples
SELECT
    ROW_NUMBER() OVER (ORDER BY sample_id)   AS sample_key,
    sample_id,
    patient_id,
    tissue_type,
    sequencing_platform,
    library_prep,
    project_id,
    CAST(collection_date AS DATE)            AS collection_date,
    ROUND(mean_coverage, 1)                  AS mean_coverage,
    ROUND(pct_bases_20x, 4)                  AS pct_bases_20x,
    CURRENT_TIMESTAMP
FROM read_csv_auto('./raw_data/samples.csv')
""")

count = con.execute('SELECT COUNT(*) FROM dim_samples').fetchone()[0]
print(f'dim_samples populated: {count:,} rows')
con.execute('SELECT * FROM dim_samples LIMIT 3').df()

dim_samples populated: 500 rows


,sample_key,sample_id,patient_id,tissue_type,sequencing_platform,library_prep,project_id,collection_date,mean_coverage,pct_bases_20x,dw_created_at
0,1,1,58,Tumor,PacBio_Sequel,WES,PROJ_002,2022-07-22,60.2,0.9136,2026-03-28 11:27:45.609659
1,2,2,217,Tumor,Illumina_NovaSeq,WGS,PROJ_001,2020-02-06,59.4,0.8466,2026-03-28 11:27:45.609659
2,3,3,288,Normal,Oxford_Nanopore,WES,PROJ_001,2021-02-05,100.9,0.7807,2026-03-28 11:27:45.609659


In [4]:
# ── dim_genes ─────────────────────────────────────────────────────────────────
con.execute("""
CREATE TABLE dim_genes (
    gene_key       INTEGER PRIMARY KEY,
    gene_id        INTEGER NOT NULL,
    gene_name      VARCHAR,
    chromosome     VARCHAR,
    strand         VARCHAR(1),
    biotype        VARCHAR,
    is_cancer_gene BOOLEAN,
    gene_length    BIGINT,          -- end_pos - start_pos
    dw_created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

con.execute("""
INSERT INTO dim_genes
SELECT
    ROW_NUMBER() OVER (ORDER BY gene_id)     AS gene_key,
    gene_id,
    gene_name,
    chromosome,
    strand,
    biotype,
    CAST(is_cancer_gene AS BOOLEAN)          AS is_cancer_gene,
    (end_pos - start_pos)                    AS gene_length,
    CURRENT_TIMESTAMP
FROM read_csv_auto('./raw_data/genes.csv')
""")

count = con.execute('SELECT COUNT(*) FROM dim_genes').fetchone()[0]
print(f'dim_genes populated: {count:,} rows')
con.execute('SELECT * FROM dim_genes LIMIT 3').df()

dim_genes populated: 100 rows


,gene_key,gene_id,gene_name,chromosome,strand,biotype,is_cancer_gene,gene_length,dw_created_at
0,1,1,TP53,chr1,+,protein_coding,True,941871,2026-03-28 11:27:47.617981
1,2,2,BRCA1,chr1,+,protein_coding,True,2117586,2026-03-28 11:27:47.617981
2,3,3,BRCA2,chrX,-,pseudogene,True,157713,2026-03-28 11:27:47.617981


## 4. Create Fact Tables

Fact tables are **wide, append-only** tables containing events and measures. They reference dimension tables via foreign keys. The grain (row-level detail) of a fact table is critical to get right.

In [5]:
# ── fact_variant_calls ────────────────────────────────────────────────────────
# Grain: one row per variant call

con.execute("""
CREATE TABLE fact_variant_calls (
    call_key      INTEGER PRIMARY KEY,
    call_id       INTEGER NOT NULL,     -- natural key
    sample_key    INTEGER REFERENCES dim_samples(sample_key),
    date_key      INTEGER REFERENCES dim_date(date_key),
    caller        VARCHAR,
    filter_status VARCHAR,
    -- Pre-computed measures
    variant_count INTEGER,
    mean_quality  DECIMAL(8,2),
    dw_created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# ETL: join source variant_calls to dimension tables to get surrogate keys
# Pre-compute variant_count and mean_quality from variants table
con.execute("""
INSERT INTO fact_variant_calls
SELECT
    ROW_NUMBER() OVER (ORDER BY vc.call_id)                           AS call_key,
    vc.call_id,
    ds.sample_key,
    CAST(strftime(CAST(vc.call_date AS DATE), '%Y%m%d') AS INTEGER)  AS date_key,
    vc.caller,
    vc.filter_status,
    COUNT(v.variant_id)                                               AS variant_count,
    ROUND(AVG(v.quality_score), 2)                                    AS mean_quality,
    CURRENT_TIMESTAMP
FROM read_csv_auto('./raw_data/variant_calls.csv') vc
JOIN dim_samples ds ON vc.sample_id = ds.sample_id
LEFT JOIN read_csv_auto('./raw_data/variants.csv') v ON vc.call_id = v.call_id
GROUP BY vc.call_id, vc.call_date, vc.caller, vc.filter_status, vc.sample_id, ds.sample_key
""")

count = con.execute('SELECT COUNT(*) FROM fact_variant_calls').fetchone()[0]
print(f'fact_variant_calls populated: {count:,} rows')
con.execute('SELECT * FROM fact_variant_calls LIMIT 3').df()

fact_variant_calls populated: 2,000 rows


,call_key,call_id,sample_key,date_key,caller,filter_status,variant_count,mean_quality,dw_created_at
0,1,1,87,20231229,Strelka2,PASS,5,42.14,2026-03-28 11:29:44.198103
1,2,2,403,20220314,Mutect2,PASS,6,44.95,2026-03-28 11:29:44.198103
2,3,3,431,20201205,DeepVariant,Artifact,3,39.30,2026-03-28 11:29:44.198103


In [6]:
# ── fact_variants ─────────────────────────────────────────────────────────────
# Grain: one row per variant (finest granularity)

con.execute("""
CREATE TABLE fact_variants (
    variant_key     INTEGER PRIMARY KEY,
    variant_id      INTEGER NOT NULL,
    call_key        INTEGER REFERENCES fact_variant_calls(call_key),
    call_id         INTEGER NOT NULL,
    sample_key      INTEGER REFERENCES dim_samples(sample_key),
    gene_key        INTEGER REFERENCES dim_genes(gene_key),
    date_key        INTEGER REFERENCES dim_date(date_key),
    variant_type    VARCHAR,
    consequence     VARCHAR,
    -- Measures
    af_tumor        DECIMAL(6,4),
    depth           INTEGER,
    quality_score   DECIMAL(6,2),
    signal_strength DECIMAL(10,2),  -- depth * af_tumor
    dw_created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

con.execute("""
INSERT INTO fact_variants
SELECT
    ROW_NUMBER() OVER (ORDER BY v.variant_id)   AS variant_key,
    v.variant_id,
    fvc.call_key,
    v.call_id,
    fvc.sample_key,
    dg.gene_key,
    fvc.date_key,
    v.variant_type,
    v.consequence,
    ROUND(v.af_tumor,       4)                  AS af_tumor,
    v.depth,
    ROUND(v.quality_score,  2)                  AS quality_score,
    ROUND(v.depth * v.af_tumor, 2)              AS signal_strength,
    CURRENT_TIMESTAMP
FROM read_csv_auto('./raw_data/variants.csv')  v
JOIN fact_variant_calls fvc ON v.call_id  = fvc.call_id
JOIN dim_genes          dg  ON v.gene_id  = dg.gene_id
""")

count = con.execute('SELECT COUNT(*) FROM fact_variants').fetchone()[0]
print(f'fact_variants populated: {count:,} rows')
con.execute('SELECT * FROM fact_variants LIMIT 3').df()

fact_variants populated: 5,000 rows


,variant_key,variant_id,call_key,call_id,sample_key,gene_key,date_key,variant_type,consequence,af_tumor,depth,quality_score,signal_strength,dw_created_at
0,1,1,388,388,210,52,20240722,Deletion,frameshift_variant,0.5168,75,47.25,38.76,2026-03-28 11:32:21.665882
1,2,2,1324,1324,424,32,20230930,SNV,missense_variant,0.5677,142,22.63,80.61,2026-03-28 11:32:21.665882
2,3,3,207,207,18,36,20220819,Insertion,missense_variant,0.2479,441,55.18,109.32,2026-03-28 11:32:21.665882


In [7]:
# Verify the warehouse schema
print('=== Warehouse Tables ===')
tables = con.execute('SHOW TABLES').fetchall()
for (tbl,) in tables:
    row_count = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<30} {row_count:>6,} rows')

=== Warehouse Tables ===
  dim_date                        2,922 rows
  dim_genes                         100 rows
  dim_samples                       500 rows
  fact_variant_calls              2,000 rows
  fact_variants                   5,000 rows


## 5. Analytical Queries

Now we can answer genomics questions using the warehouse. Notice how the star schema makes aggregation queries clean and fast — no complex joins through OLTP tables.

In [8]:
# ── Query 1: Monthly Variant Call Volume Trend (PASS only) ────────────────────
monthly = con.execute("""
    SELECT
        dd.year,
        dd.month,
        dd.month_name,
        COUNT(DISTINCT fvc.call_id)              AS call_count,
        SUM(fvc.variant_count)                   AS total_variants,
        COUNT(DISTINCT fvc.sample_key)           AS unique_samples
    FROM fact_variant_calls fvc
    JOIN dim_date           dd  ON fvc.date_key = dd.date_key
    WHERE fvc.filter_status = 'PASS'
    GROUP BY dd.year, dd.month, dd.month_name
    ORDER BY dd.year, dd.month
""").df()

print('=== Monthly Variant Call Volume (PASS only) ===')
print(monthly.to_string(index=False))

=== Monthly Variant Call Volume (PASS only) ===
 year  month month_name  call_count  total_variants  unique_samples
 2020      1    January           2             4.0               2
 2020      3      March           5            10.0               5
 2020      4      April           3            10.0               3
 2020      5        May           5            13.0               5
 2020      6       June           9            18.0               8
 2020      7       July           8            19.0               8
 2020      8     August           9            18.0               8
 2020      9  September          15            38.0              13
 2020     10    October          22            39.0              21
 2020     11   November          23            64.0              20
 2020     12   December          20            49.0              19
 2021      1    January          18            41.0              16
 2021      2   February          12            36.0              11


In [9]:
# ── Query 2: Top 10 Genes by Variant Count (PASS only) ────────────────────────
top_genes = con.execute("""
    SELECT
        dg.gene_name,
        dg.biotype,
        dg.is_cancer_gene,
        COUNT(fv.variant_id)            AS total_variants,
        COUNT(DISTINCT fv.sample_key)   AS unique_samples,
        ROUND(AVG(fv.af_tumor), 4)      AS mean_af_tumor
    FROM fact_variants      fv
    JOIN dim_genes          dg  ON fv.gene_key  = dg.gene_key
    JOIN fact_variant_calls fvc ON fv.call_key  = fvc.call_key
    WHERE fvc.filter_status = 'PASS'
    GROUP BY dg.gene_name, dg.biotype, dg.is_cancer_gene
    ORDER BY total_variants DESC
    LIMIT 10
""").df()

print('=== Top 10 Genes by Variant Count (PASS only) ===')
print(top_genes.to_string(index=False))

=== Top 10 Genes by Variant Count (PASS only) ===
gene_name        biotype  is_cancer_gene  total_variants  unique_samples  mean_af_tumor
     JAK3     pseudogene           False              34              34         0.5019
     SRC3 protein_coding            True              33              30         0.4764
     RAS4          miRNA           False              33              29         0.5133
    FOXO2 protein_coding            True              32              31         0.4854
     MET4     pseudogene           False              31              31         0.4755
    ERBB3 protein_coding           False              31              30         0.5438
     PTEN         lncRNA            True              31              30         0.5058
    FOXO4         lncRNA           False              31              30         0.4543
    SMAD4 protein_coding            True              31              29         0.5650
    STAT4         lncRNA           False              31              

In [10]:
# ── Query 3: Sample Mutation Burden with Window Functions ─────────────────────
burden = con.execute("""
    WITH sample_stats AS (
        SELECT
            ds.sample_key,
            ds.sample_id,
            ds.tissue_type,
            ds.project_id,
            ds.mean_coverage,
            COUNT(fv.variant_id)           AS total_variants,
            ROUND(AVG(fv.af_tumor), 4)     AS mean_af_tumor,
            ROUND(AVG(fv.depth),    1)     AS mean_depth
        FROM dim_samples        ds
        JOIN fact_variants      fv  ON ds.sample_key = fv.sample_key
        JOIN fact_variant_calls fvc ON fv.call_key   = fvc.call_key
        WHERE fvc.filter_status = 'PASS'
        GROUP BY ds.sample_key, ds.sample_id, ds.tissue_type, ds.project_id, ds.mean_coverage
    )
    SELECT
        *,
        ROUND(PERCENT_RANK() OVER (ORDER BY total_variants) * 100, 1) AS burden_percentile,
        CASE
            WHEN PERCENT_RANK() OVER (ORDER BY total_variants) >= 0.75 THEN 'High Burden'
            WHEN PERCENT_RANK() OVER (ORDER BY total_variants) >= 0.25 THEN 'Medium Burden'
            ELSE 'Low Burden'
        END AS burden_segment
    FROM sample_stats
    ORDER BY total_variants DESC
    LIMIT 15
""").df()

print('=== Top 15 Samples by Mutation Burden ===')
print(burden[['sample_id', 'tissue_type', 'project_id', 'total_variants',
              'mean_af_tumor', 'burden_percentile', 'burden_segment']].to_string(index=False))

=== Top 15 Samples by Mutation Burden ===
 sample_id tissue_type project_id  total_variants  mean_af_tumor  burden_percentile burden_segment
         9       Blood   PROJ_002              23         0.4532              100.0    High Burden
       291       Tumor   PROJ_004              19         0.5477               99.3    High Burden
       413      Normal   PROJ_003              19         0.5419               99.3    High Burden
       104       Tumor   PROJ_002              19         0.5076               99.3    High Burden
       497       Tumor   PROJ_003              18         0.5724               98.6    High Burden
       247       Blood   PROJ_008              18         0.4705               98.6    High Burden
       183        PBMC   PROJ_006              18         0.4249               98.6    High Burden
        28       Tumor   PROJ_001              16         0.5054               98.1    High Burden
       110       Blood   PROJ_005              16         0.5019   

In [11]:
# ── Query 4: Variant Counts by Tissue Type and Project ────────────────────────
by_tissue = con.execute("""
    SELECT
        ds.tissue_type,
        ds.project_id,
        COUNT(DISTINCT ds.sample_key)   AS sample_count,
        COUNT(fv.variant_id)            AS total_variants,
        ROUND(AVG(fv.af_tumor), 4)      AS mean_af_tumor
    FROM dim_samples        ds
    JOIN fact_variants      fv  ON ds.sample_key = fv.sample_key
    JOIN fact_variant_calls fvc ON fv.call_key   = fvc.call_key
    WHERE fvc.filter_status = 'PASS'
    GROUP BY ds.tissue_type, ds.project_id
    ORDER BY ds.tissue_type, total_variants DESC
""").df()

print('=== Variant Counts by Tissue Type and Project ===')
print(by_tissue.to_string(index=False))

=== Variant Counts by Tissue Type and Project ===
tissue_type project_id  sample_count  total_variants  mean_af_tumor
      Blood   PROJ_005            12              87         0.4866
      Blood   PROJ_006            13              84         0.4753
      Blood   PROJ_008            10              80         0.5033
      Blood   PROJ_002            11              76         0.4800
      Blood   PROJ_007            12              70         0.4983
      Blood   PROJ_001             9              68         0.5296
      Blood   PROJ_003            11              51         0.4926
      Blood   PROJ_004             9              39         0.5413
     Normal   PROJ_003            12              80         0.4834
     Normal   PROJ_005            12              63         0.4541
     Normal   PROJ_007            13              56         0.4952
     Normal   PROJ_002             9              50         0.5255
     Normal   PROJ_006             7              48         0.483

In [12]:
# ── Query 5: Consequence Distribution with Quarter-over-Quarter Change ─────────
cons_qoq = con.execute("""
    WITH quarterly AS (
        SELECT
            fv.consequence,
            dd.year,
            dd.quarter,
            COUNT(fv.variant_id) AS variant_count
        FROM fact_variants      fv
        JOIN dim_date           dd  ON fv.date_key  = dd.date_key
        JOIN fact_variant_calls fvc ON fv.call_key  = fvc.call_key
        WHERE fvc.filter_status = 'PASS'
        GROUP BY fv.consequence, dd.year, dd.quarter
    )
    SELECT
        consequence,
        year,
        quarter,
        variant_count,
        LAG(variant_count) OVER (PARTITION BY consequence ORDER BY year, quarter) AS prev_quarter_count,
        ROUND(
            (variant_count - LAG(variant_count) OVER (PARTITION BY consequence ORDER BY year, quarter))
            / NULLIF(LAG(variant_count) OVER (PARTITION BY consequence ORDER BY year, quarter), 0) * 100,
        1) AS qoq_growth_pct
    FROM quarterly
    ORDER BY consequence, year, quarter
""").df()

print('=== Consequence Distribution with QoQ Change ===')
print(cons_qoq.to_string(index=False))

=== Consequence Distribution with QoQ Change ===
        consequence  year  quarter  variant_count  prev_quarter_count  qoq_growth_pct
 frameshift_variant  2020        1              1                <NA>             NaN
 frameshift_variant  2020        2              8                   1           700.0
 frameshift_variant  2020        3             15                   8            87.5
 frameshift_variant  2020        4             29                  15            93.3
 frameshift_variant  2021        1             25                  29           -13.8
 frameshift_variant  2021        2             28                  25            12.0
 frameshift_variant  2021        3             32                  28            14.3
 frameshift_variant  2021        4             39                  32            21.9
 frameshift_variant  2022        1             24                  39           -38.5
 frameshift_variant  2022        2             24                  24             0.0
 fram

## 6. Query Execution Plans with EXPLAIN

`EXPLAIN` shows you how DuckDB plans to execute a query — which operators it uses and in what order. This is essential for query optimization.

In [13]:
# Use EXPLAIN to see the query plan for the tissue type aggregation
plan = con.execute("""
EXPLAIN
SELECT
    ds.tissue_type,
    COUNT(fv.variant_id)       AS total_variants,
    ROUND(AVG(fv.af_tumor), 4) AS mean_af_tumor
FROM dim_samples        ds
JOIN fact_variants      fv  ON ds.sample_key = fv.sample_key
JOIN fact_variant_calls fvc ON fv.call_key   = fvc.call_key
WHERE fvc.filter_status = 'PASS'
GROUP BY ds.tissue_type
ORDER BY total_variants DESC
""").fetchall()

print('=== DuckDB Query Execution Plan ===')
for row in plan:
    print(row[1])

=== DuckDB Query Execution Plan ===
┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│           g(#0)           │
│             #1            │
│             #2            │
│                           │
│          ~0 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│ count(fv.variant_id) DESC │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_compress_string_│
│        ubigint(#0)        │
│             #1            │
│             #2            │
│                           │
│          ~5 rows          │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│        tissue_type        │
│       total_variants      │
│       mean_af_tumor       │
│                           │
│   

In [ ]:
# EXPLAIN ANALYZE adds actual row counts and timing to each operator
analyze = con.execute("""
EXPLAIN ANALYZE
SELECT
    dg.biotype,
    COUNT(fv.variant_id)       AS total_variants
FROM fact_variants      fv
JOIN dim_genes          dg  ON fv.gene_key  = dg.gene_key
JOIN fact_variant_calls fvc ON fv.call_key  = fvc.call_key
WHERE fvc.filter_status = 'PASS'
GROUP BY dg.biotype
""").fetchall()

print('=== Execution Analysis (with actual timings) ===')
for row in analyze:
    print(row[1])

In [ ]:
# Final warehouse summary
print('=== Genomics Data Warehouse Summary ===')
print(f'Database file: {DB_PATH}')
print(f'File size:     {os.path.getsize(DB_PATH)/1024:.1f} KB')
print()
print('Tables:')
for (tbl,) in con.execute('SHOW TABLES').fetchall():
    cnt = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<30} {cnt:>6,} rows')

con.close()
print('\nDatabase connection closed.')

## Exercises

**Exercise 3.1 — Cohort Analysis**

A cohort analysis groups samples by the month their first variant call was processed, then tracks how many samples from each cohort had variant calls in subsequent months.

Write a DuckDB query that produces a cohort table with:
- `cohort_year`, `cohort_month` — when the sample first had a PASS call
- `months_since_first_call` — 0, 1, 2, ...
- `samples_active` — how many samples from that cohort had a call in that relative month
- `variant_count` — total PASS variants from that cohort in that relative month

Hint: Use `MIN(dd.full_date) OVER (PARTITION BY sample_key)` to find each sample's first call date, then compute the difference in months.

In [ ]:
# Reopen connection for exercises
con = duckdb.connect('./warehouse/genomics.duckdb')

# Exercise 3.1 — Cohort Analysis
cohort = con.execute("""
    -- TODO: write cohort analysis query
    SELECT 'replace me' AS cohort_year
""").df()
print(cohort)

**Exercise 3.2 — Running Total and Moving Average**

Write a query that produces a monthly time series of PASS variant counts with:
- `year`, `month`
- `monthly_variants`
- `cumulative_variants` — running total of variants from the start
- `rolling_3mo_avg` — 3-month moving average of monthly variant counts

Use DuckDB window functions (`SUM() OVER`, `AVG() OVER` with a ROWS frame).

In [ ]:
# Exercise 3.2 — Running Total and Moving Average
rolling = con.execute("""
    -- TODO: write your query
    SELECT 'replace me' AS placeholder
""").df()
print(rolling)

**Exercise 3.3 — Gene Co-occurrence Analysis**

Which pairs of genes are most frequently mutated in the **same variant call** (same `call_id`)? This is analogous to a market basket analysis — genes that co-occur may be in the same pathway.

Write a query that returns the top 10 gene pairs (by co-occurrence count). Each row should have:
- `gene_name_a`
- `gene_name_b`
- `co_occurrence_count`

Hint: Self-join `fact_variants` on `call_id` where `gene_key_a < gene_key_b` to avoid duplicate pairs.

In [ ]:
# Exercise 3.3 — Gene Co-occurrence
cooccurrence = con.execute("""
    -- TODO: write your query
    SELECT 'replace me' AS placeholder
""").df()
print(cooccurrence)

con.close()

---

**Next:** `04_lakehouse.ipynb` — Building a local lakehouse with Delta Lake